# Exploration des données

Premier regard sur les tables sorties de la chaîne de traitement : les ventes, avec leur DPE quand il a été retrouvé, la table par commune et les indicateurs de l'Insee.

Bibliothèques nécessaires : `python -m pip install -r requirements.txt matplotlib`

## Paramètres

Par défaut, on charge la France entière (3,2 millions de ventes, environ 0,6 Go de mémoire). Pour un essai rapide, mettre un code de département, par exemple `"44"`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

DEPARTEMENT = None  # None = France entière ; "44" pour un essai sur un département

## Ventes

`charger_ventes` lit le fichier, ajoute l'année et le trimestre de vente, et range les étiquettes de A à G. `echantillon_analyse` garde les ventes appariées, sans les ventes rapprochées d'un même logement.

In [ ]:
ventes = chargement.charger_ventes(DEPARTEMENT)
appariees = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes, dont {ventes['apparie'].sum():,} appariées")
print(f"{ventes['vente_rapprochee'].sum():,} ventes rapprochées, exclues de l'analyse des prix")
print(f"échantillon d'analyse : {len(appariees):,} ventes")

In [ ]:
ventes.head()

In [ ]:
ventes[["prix", "surface", "nb_pieces", "prix_m2", "conso_5_usages_par_m2_ep", "delai_jours"]].describe()

### Valeurs manquantes

Les colonnes du DPE sont vides quand la vente n'est pas appariée.

In [ ]:
(ventes.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame("% manquant")

## Taux d'appariement

Plus bas en 2022 : une partie des ventes se faisait encore avec un DPE établi avant juillet 2021, absent de la base de l'ADEME.

In [ ]:
taux = (ventes.groupby(["annee", "type_bien"], observed=True)["apparie"].mean() * 100).unstack().round(1)
display(taux)

ax = taux.plot(kind="bar", rot=0, color=["#14213D", "#A23B3B"])
ax.set(title="Ventes appariées à un DPE (%)", xlabel="", ylabel="%")
plt.show()

In [ ]:
par_dep = (ventes.groupby("code_departement", observed=True)["apparie"].mean() * 100).round(1).sort_values()
print("Départements au taux le plus bas :"); display(par_dep.head(5).to_frame("% apparié"))
print("Départements au taux le plus haut :"); display(par_dep.tail(5).to_frame("% apparié"))

In [ ]:
display(appariees["methode_appariement"].value_counts(normalize=True).mul(100).round(1).to_frame("%"))
print("part ambiguë :", round(appariees["ambigu"].mean() * 100, 1), "%")

## Répartition des étiquettes (échantillon d'analyse)

In [ ]:
rep = appariees["etiquette_dpe"].value_counts(normalize=True).reindex(ORDRE).mul(100)
ax = rep.plot(kind="bar", rot=0, color=[COULEURS[e] for e in ORDRE], edgecolor="black", linewidth=0.4)
ax.set(title="Étiquettes DPE des logements vendus (%)", xlabel="", ylabel="%")
for i, v in enumerate(rep):
    ax.text(i, v + 0.4, f"{v:.0f}", ha="center")
plt.show()

In [ ]:
pd.crosstab(appariees["type_bien"], appariees["etiquette_dpe"], normalize="index").mul(100).round(1)

## Prix au m² par étiquette

Écarts bruts : ils mélangent l'effet de l'étiquette avec la localisation, l'âge du logement, etc. La comparaison à logement comparable est faite dans `verification_appariement.ipynb`, puis dans le rapport.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, type_bien in zip(axes, ["appartement", "maison"]):
    sous = appariees[appariees["type_bien"] == type_bien]
    groupes = [sous.loc[sous["etiquette_dpe"] == e, "prix_m2"].dropna() for e in ORDRE]
    boites = ax.boxplot(groupes, showfliers=False, patch_artist=True)
    ax.set_xticks(range(1, len(ORDRE) + 1), ORDRE)
    for boite, e in zip(boites["boxes"], ORDRE):
        boite.set_facecolor(COULEURS[e])
    ax.set(title=type_bien, xlabel="", ylabel="€/m²")
plt.suptitle("Prix au m² selon l'étiquette (hors valeurs extrêmes)")
plt.tight_layout()
plt.show()

In [ ]:
(appariees.groupby(["type_bien", "etiquette_dpe"], observed=True)["prix_m2"]
    .describe(percentiles=[0.25, 0.5, 0.75])[["count", "25%", "50%", "75%"]]
    .rename(columns={"count": "n", "50%": "médiane"}))

## Évolution 2022-2025

Le taux d'appariement n'est pas le même selon les années : à lire avec prudence.

In [ ]:
evol = appariees.groupby(["annee", "etiquette_dpe"], observed=True)["prix_m2"].median().unstack()
ax = evol.plot(marker="o", color=[COULEURS[e] for e in evol.columns])
ax.set(title="Prix médian au m²", xlabel="", ylabel="€/m²")
ax.set_xticks(evol.index)
ax.legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.show()

## Prix et consommation

Pointillés : seuils de consommation entre lettres (70, 110, 180, 250, 330, 420 kWhEP/m²/an). La lettre dépend aussi des émissions de gaz à effet de serre, d'où des points qui dépassent les seuils.

In [ ]:
echantillon = appariees.dropna(subset=["conso_5_usages_par_m2_ep", "prix_m2"])
echantillon = echantillon.sample(min(20_000, len(echantillon)), random_state=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(echantillon["conso_5_usages_par_m2_ep"], echantillon["prix_m2"], s=3, alpha=0.25,
           c=echantillon["etiquette_dpe"].map(COULEURS).astype(str))
for seuil in [70, 110, 180, 250, 330, 420]:
    ax.axvline(seuil, color="grey", lw=0.8, ls="--")
ax.set(xlim=(0, 700), ylim=(0, echantillon["prix_m2"].quantile(0.99)),
       xlabel="consommation (kWhEP/m²/an)", ylabel="€/m²", title="Prix au m² et consommation (échantillon)")
plt.show()

## Carte rapide

Des points placés par leurs coordonnées, sans fond de carte, sur un échantillon de ventes. Les cartes par département du rapport utiliseront `geopandas`.

In [ ]:
carte = appariees.dropna(subset=["longitude", "latitude"])
carte = carte.sample(min(30_000, len(carte)), random_state=1)

fig, ax = plt.subplots(figsize=(7, 7))
for e in ORDRE:
    pts = carte[carte["etiquette_dpe"] == e]
    ax.scatter(pts["longitude"], pts["latitude"], s=2, color=COULEURS[e], label=e, alpha=0.6)
ax.set_aspect(1 / 0.7)  # à peu près cos(latitude) en France
ax.set(xlabel="longitude", ylabel="latitude")
ax.legend(markerscale=5, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

## Table par commune

Une ligne par commune, année et type de bien. `part_FG` et `part_AB` sont calculées sur les DPE de la commune.

In [ ]:
communes = pd.read_parquet(config.PROCESSED / "communes_france.parquet")
if DEPARTEMENT:
    communes = communes[communes["code_commune"].str.startswith(DEPARTEMENT)]
communes.head()

In [ ]:
derniere = communes[(communes["annee"] == communes["annee"].max()) & (communes["n_ventes"] >= 10)]
fig, ax = plt.subplots()
for type_bien, couleur in [("appartement", "#14213D"), ("maison", "#A23B3B")]:
    s = derniere[derniere["type_bien"] == type_bien]
    ax.scatter(s["part_FG"] * 100, s["prix_m2_median"], s=8, alpha=0.5, color=couleur, label=type_bien)
ax.set(xlabel="part de F-G (%)", ylabel="prix médian (€/m²)",
       title=f"Communes, {derniere['annee'].max()} (au moins 10 ventes)")
ax.legend(frameon=False)
plt.show()

## Indicateurs de l'Insee

Libellés des codes dans `reports/insee_dictionnaire.csv` (par exemple `MED_SL` pour le niveau de vie médian). Le niveau de vie n'est pas diffusé pour les très petites communes (secret statistique).

In [ ]:
insee = pd.read_parquet(config.PROCESSED / "insee_communes.parquet")
dico = pd.read_csv(config.REPORTS / "insee_dictionnaire.csv")
display(dico)
insee.head()

In [ ]:
communes_insee = communes.merge(insee[["code_commune", "MED_SL", "POP"]], on="code_commune", how="left")
print("MED_SL manquant :", round(communes_insee["MED_SL"].isna().mean() * 100, 1), "%")
communes_insee.head()